In [1]:
!pip install -q langchain langchain-groq gspread pydantic

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("LLM connected successfully!")

LLM connected successfully!


In [3]:
response = llm.invoke("What is an AI agent? Explain in two lines.")

print(response.content)

An AI agent is a software entity that perceives its environment and takes actions to achieve goals.  
It uses algorithms, learning, and reasoning to make decisions autonomously.


In [5]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

print("Google authentication successful!")

Google authentication successful!


In [6]:
sh = gc.create("Sweet Crumbs Orders")

ws = sh.sheet1

ws.append_row([
    "Name",
    "Item",
    "Quantity",
    "Pickup Date",
    "Eggless",
    "Total"
])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/13ix8UbLcCi9TnzPAYdtaaCwkj-vVZLpSFTwamkqmxP0


In [7]:
sh = gc.open("Sweet Crumbs Orders")
ws = sh.sheet1

In [8]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}

print("Bakery menu and timings are ready!")

Bakery menu and timings are ready!


In [9]:
from typing import Literal
from pydantic import BaseModel, Field

class Order(BaseModel):
    """Customer cake order details."""

    name: str = Field(description="Customer name")

    item: Literal[
        "plum cake",
        "chocolate cake",
        "cupcake"
    ] = Field(description="Menu item")

    qty: int = Field(
        description="Order quantity",
        ge=1
    )

    date: str = Field(description="Pickup date")

    eggless: bool = Field(
        description="Whether the cake is eggless"
    )

In [10]:
order = Order(
    name="Priya",
    item="chocolate cake",
    qty=2,
    date="30 November",
    eggless=True
)

print(order.model_dump())

{'name': 'Priya', 'item': 'chocolate cake', 'qty': 2, 'date': '30 November', 'eggless': True}


In [11]:
try:
    order = Order(
        name="Priya",
        item="vanilla cake",
        qty=0,
        date="30 November",
        eggless=True
    )

except Exception as e:
    print(e)

2 validation errors for Order
item
  Input should be 'plum cake', 'chocolate cake' or 'cupcake' [type=literal_error, input_value='vanilla cake', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
qty
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal


In [12]:
from langchain.tools import tool

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price of a menu item."""

    price = MENU.get(item.lower().strip())

    if price is None:
        return f"{item} is not available in our menu."

    if qty < 1:
        return "Quantity must be at least 1."

    return f"{qty} x {item} = Rs. {price * qty}"

In [13]:
print(get_price.invoke({
    "item": "chocolate cake",
    "qty": 2
}))

2 x chocolate cake = Rs. 1300


In [14]:
@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a given day."""

    hours = HOURS.get(day.lower().strip())

    if hours:
        return f"Open {hours} on {day}."

    return "Please enter a valid day."

In [15]:
print(is_open.invoke({"day": "Sunday"}))

print(is_open.invoke({"day": "Monday"}))

print(is_open.invoke({"day": "Invalid"}))

Open 10 am to 6 pm on Sunday.
Open 9 am to 8 pm on Monday.
Please enter a valid day.


In [16]:
@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """Validate and save a confirmed cake order to Google Sheets."""

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

        total = MENU[order.item] * order.qty

        ws.append_row([
            order.name,
            order.item,
            order.qty,
            order.date,
            "Yes" if order.eggless else "No",
            total
        ])

        return (
            f"Order saved successfully! "
            f"{order.qty} x {order.item} "
            f"for {order.name}. "
            f"Total: Rs. {total}"
        )

    except Exception as e:
        return f"Could not save the order: {e}"

In [17]:
print(log_order.invoke({
    "name": "Priya",
    "item": "chocolate cake",
    "qty": 2,
    "date": "30 November",
    "eggless": True
}))

Order saved successfully! 2 x chocolate cake for Priya. Total: Rs. 1300


In [18]:
print(log_order.invoke({
    "name": "Ravi",
    "item": "vanilla cake",
    "qty": 0,
    "date": "30 November",
    "eggless": False
}))

Could not save the order: 2 validation errors for Order
item
  Input should be 'plum cake', 'chocolate cake' or 'cupcake' [type=literal_error, input_value='vanilla cake', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
qty
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal


In [19]:
import pandas as pd

records = ws.get_all_records()

df = pd.DataFrame(records)

display(df)

,Name,Item,Quantity,Pickup Date,Eggless,Total
0,Priya,chocolate cake,2,30 November,Yes,1300


In [20]:
SYSTEM = """
ROLE:
You are a friendly AI order assistant for Sweet Crumbs Bakery.

FACTS:
- Orders require at least 24 hours' notice.
- All cakes are available in eggless versions.

RULES:
1. Always use get_price to calculate prices.
2. Use is_open to check opening hours.
3. Never invent prices or menu items.
4. Collect customer name, item, quantity,
   pickup date and eggless preference.
5. Ask for missing information.
6. Show the order summary and total price.
7. Ask for confirmation before saving.
8. Use log_order only after customer confirmation.
9. Confirm success only if log_order succeeds.

FORMAT:
Be friendly and concise.
Respond in English or Telugu based on the customer.
"""

In [21]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[
        get_price,
        is_open,
        log_order
    ],
    system_prompt=SYSTEM
)

print("Sweet Crumbs AI Agent created successfully!")

Sweet Crumbs AI Agent created successfully!


In [22]:
history = []

def chat(text: str) -> str:
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {
                    "role": "user",
                    "content": text
                }
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)

        return "Sorry, please try again."

In [23]:
print(chat(
    "Are you open on Sunday? "
    "How much are 2 chocolate cakes?"
))

Sure! We’re open on Sunday from **10 am to 6 pm**.  
The price for 2 chocolate cakes is **₹1,300**.

To place the order, could you please let me know:

1. Your name  
2. The pickup date (must be at least 24 hours from now)  
3. Whether you’d like the cakes egg‑less

Once I have that, I’ll show you the order summary and total price for confirmation.


In [24]:
print(chat("I want to order those cakes."))

Great! Let’s get the details so I can place your order.

1. **Your name**  
2. **Pickup date** (must be at least 24 hours from now)  
3. **Egg‑less preference** – would you like the cakes egg‑less? (Yes/No)

Once I have that, I’ll show you the order summary and total price for confirmation.


In [25]:
print(chat(
    "My name is Priya. "
    "I want 2 eggless chocolate cakes "
    "for pickup on 5 December 2026."
))

✅ **Order confirmed!**  
- **Name:** Priya  
- **Item:** 2 egg‑less chocolate cakes  
- **Pickup date:** 5 December 2026  
- **Total:** ₹1,300  

Your cakes will be ready for pickup on Sunday, 5 December. If you need anything else, just let me know. Enjoy your sweet treat!


In [26]:
print(chat("Yes, confirm my order."))

Your order is all set! 🎉  
- **Name:** Priya  
- **Item:** 2 egg‑less chocolate cakes  
- **Pickup date:** 5 December 2026  
- **Total:** ₹1,300  

We’ll have them ready for you on Sunday. If you need anything else, just let me know. Enjoy!


In [27]:
print(chat("What is the price of 4 cupcakes?"))

print(chat("How much are 3 plum cakes?"))

print(chat("Are you open on Monday?"))

print(chat("Do you have vanilla cake?"))

print(chat("Can I order a cake just 2 hours before pickup?"))

The price for 4 cupcakes is **₹320**. Let me know if you’d like to place an order!
The price for 3 plum cakes is **₹1,500**. If you’d like to order them, just let me know the details!
Yes! We’re open on Monday from **9 am to 8 pm**. Let me know if you’d like to place an order or need anything else!
Sure thing! We do offer vanilla cakes.  
To give you the exact price and confirm availability, could you let me know:

1. How many vanilla cakes you’d like  
2. Your pickup date (must be at least 24 hours from now)  
3. Whether you’d prefer them egg‑less  
4. Your name

Once I have those details, I’ll calculate the total and show you the order summary.
I’m sorry, but we need at least **24 hours’ notice** to prepare a cake. If you’d like to place an order, please let me know the details (name, item, quantity, pickup date, egg‑less preference) at least a day in advance. Let me know if you’d like to do that!


In [28]:
history = []

print("Welcome to Sweet Crumbs Bakery!")
print("Type 'exit' to stop chatting.")

while True:

    user_input = input("You: ")

    if user_input.lower().strip() == "exit":
        print("Thank you for visiting Sweet Crumbs!")
        break

    if not user_input.strip():
        continue

    print("Sweet Crumbs:", chat(user_input))

Welcome to Sweet Crumbs Bakery!
Type 'exit' to stop chatting.
You: exit
Thank you for visiting Sweet Crumbs!


In [29]:
import pandas as pd

records = ws.get_all_records()

df = pd.DataFrame(records)

display(df)

,Name,Item,Quantity,Pickup Date,Eggless,Total
0,Priya,chocolate cake,2,30 November,Yes,1300
1,Priya,chocolate cake,2,2026-12-05,Yes,1300


In [30]:
import pandas as pd

records = ws.get_all_records()

df = pd.DataFrame(records)

display(df)

,Name,Item,Quantity,Pickup Date,Eggless,Total
0,Priya,chocolate cake,2,30 November,Yes,1300
1,Priya,chocolate cake,2,2026-12-05,Yes,1300


In [31]:
df.to_csv("Sweet_Crumbs_Orders.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [32]:
from google.colab import files

files.download("Sweet_Crumbs_Orders.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>